In [14]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from src.db_connect import get_db_engine
from sqlalchemy import text
import sys
sys.path.append('..')
%matplotlib inline


In [ ]:
# CONNECTING TO THE DATABASE
engine = get_db_engine()

try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
        print("Connected Successfully!")
except Exception as e:
    print(f"Connection Failed: {e}")


Connected Successfully!


In [ ]:
# LOADING THE DATA FROM THE DATABASE
df = pd.read_sql("SELECT * FROM creditcarddb.credit_card_info", con = engine)

df.head()

,id,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [15]:
# HANDLING THE NULL VALUES IN THE "Monthly_Income" COLUMN

df["MonthlyIncome"] =  df["MonthlyIncome"].replace(0, np.nan)
df['MonthlyIncome'].isnull().sum()

np.int64(31365)

## Step 2 — Drop Unnecessary Columns
Dropping id (row number) and two highly correlated late payment 
columns identified during EDA (0.98-0.99 correlation).
Keeping NumberOfTimes90DaysLate as it represents most severe default signal.

In [20]:
df =  df.drop(columns =["id",
                        "NumberOfTime30-59DaysPastDueNotWorse",
                        "NumberOfTime60-89DaysPastDueNotWorse"])    

In [24]:
df.columns


Index(['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age',
       'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
       'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines',
       'NumberOfDependents'],
      dtype='object')

## Step 3 - Handling the NULL values in MonthlyIncome

In [27]:
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())

In [28]:
df["MonthlyIncome"].isnull().sum()

np.int64(0)

## Step 4 - Handling Outliers

In [33]:
# Filtering the bad rows from the data set
# Current df → 150,000 rows (including 269 bad rows)
# After filter → 149,731 rows (bad rows removed)

df = df[~df["NumberOfTimes90DaysLate"].isin([96,98])]
df.shape

(149731, 9)

In [34]:
# REMOVING THE ONE UNDERAGE ROW WHICH IS A OUTLIER
df = df[df["age"] > 18]
df.shape

(149730, 9)

In [36]:
df["RevolvingUtilizationOfUnsecuredLines"] = df["RevolvingUtilizationOfUnsecuredLines"].clip(upper = 1)
df["RevolvingUtilizationOfUnsecuredLines"].max()

np.float64(1.0)

In [ ]:
# IDENTIFYING THE EXTREM OUTLIEARS IN "DebtRatio" columns



df["DebtRatio"] = df["DebtRatio"].clip(upper = 10)

df['DebtRatio'].max()

np.float64(10.0)

## Step 5 — SMOTE

In [44]:
x = df.drop(columns =["SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]
print(x.shape)
print(y.shape)
print(y.value_counts())

(149730, 8)
(149730,)
SeriousDlqin2yrs
0    139851
1      9879
Name: count, dtype: int64


In [46]:
# x = feature and y = target
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify= y)

# Apply SMOTE only on training data
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train,y_train)

print(X_train_res.shape)
print(y_train_res.value_counts())

(223762, 8)
SeriousDlqin2yrs
0    111881
1    111881
Name: count, dtype: int64


## Step 6 — Save Processed Data
- Saving train and test sets separately.
- SMOTE applied only to training data.
- Test data kept original for unbiased model evaluation.

In [47]:
X_train_res.to_csv("../data/processed_data/X_train.csv", index = False)
y_train_res.to_csv("../data/processed_data/y_train.csv", index = False)
X_test.to_csv("../data/processed_data/X_test.csv", index = False)
y_test.to_csv("../data/processed_data/y_test.csv", index = False)